# Rebuild BLIP + ALIGN Gallery Embeddings (GPU)

**Problem:** The committed `blip_image_embeddings.npz` and `align_image_embeddings.npz` are corrupt — re-encoding the same image gives cosine 0.22–0.59 against the stored row (should be 1.0000). Root cause: the old build iterated over all 8,091 image paths and dropped missing files silently, leaving the saved 1,600 rows in an unknown order.

**Result:** BLIP P@5 = 0.0% in `precision_scores.json` (looks like BLIP is broken).

**Fix:** Re-encode the 1,600 captioned images (same order as `metadata.npz:image_names`) on Colab T4, save as float32 with `image_names` inside each npz, then download.

**Runtime on T4:** BLIP ~1 min, ALIGN ~3 min.

## 1. Setup — install deps, enable GPU

Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
!pip install -q "open_clip_torch==3.3.0" "torch>=2.0" torchvision pillow numpy
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
import open_clip; print('open_clip', open_clip.__version__)
# Must say 'CUDA available: True' before continuing.

## 2. Get the project files into Colab

Upload just the **3 files** the rebuild needs (skip `model.py` — the rebuild only imports `model_comparison`):

- `metadata.npz` (from `embeddings/`)
- `rebuild_comparison_embeddings.py`
- `model_comparison.py`

In [ ]:
import os
os.makedirs('embeddings', exist_ok=True)

from google.colab import files
uploaded = files.upload()  # pick the 3 files listed above

# metadata.npz must land in embeddings/ (the script expects it there)
if 'metadata.npz' in uploaded:
    os.replace('metadata.npz', 'embeddings/metadata.npz')
    print('metadata.npz placed at embeddings/metadata.npz')
print('Files in cwd:', [f for f in os.listdir('.') if f.endswith(('.py','.npz'))])

## 3. Download Flickr8k images (direct, with real filenames)

We need the 1,600 captioned images with their **original filenames** (e.g. `1000268201_693b08cb0e.jpg`) so they line up with `metadata.npz:image_names`. The HF `tsystems/flickr8k` dataset drops the filename column, so we download the raw zip from the standard GitHub mirror instead. This gives us the exact same files your local `data/` folder has.

In [ ]:
# Download Flickr8k images (~1 GB) from the standard GitHub mirror.
!wget -q --show-progress "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip" -O Flickr8k_Dataset.zip
!unzip -q -o Flickr8k_Dataset.zip -d /tmp/flickr8k_extract

# The zip was created on macOS, so it spams '._<name>.jpg' AppleDouble junk
# files and a '__MACOSX/' folder. Delete them or the gallery check fails.
!find /tmp/flickr8k_extract -name '._*' -delete
!rm -rf /tmp/flickr8k_extract/__MACOSX
print('macOS junk cleaned.')

# Now locate the directory that holds the REAL images (no '._' prefix).
import subprocess, os, glob
all_dirs = subprocess.check_output(
    "find /tmp/flickr8k_extract -type d", shell=True).decode().strip().splitlines()
real_img_dirs = []
for d in all_dirs:
    jpgs = [f for f in os.listdir(d)
            if f.lower().endswith('.jpg') and not f.startswith('._')]
    if len(jpgs) > 1000:
        real_img_dirs.append((d, len(jpgs)))
assert real_img_dirs, 'No directory with >1000 real .jpg files found after cleanup!'
SRC_DIR, n_files = real_img_dirs[0]
print(f'Image source dir: {SRC_DIR}  ({n_files} real .jpg files)')

# Symlink it to the path the script's resolver expects.
DST_DIR = 'data/Flickr8k_Dataset/Flicker8k_Dataset'
os.makedirs('data/Flickr8k_Dataset', exist_ok=True)
if os.path.lexists(DST_DIR):
    os.remove(DST_DIR)
os.symlink(os.path.abspath(SRC_DIR), DST_DIR)
print(f'Symlinked {DST_DIR} -> {SRC_DIR}')
print('Verify:', len(os.listdir(DST_DIR)), 'files in', DST_DIR)
print('Sample:', sorted(os.listdir(DST_DIR))[:3])

In [ ]:
# Verify the 1,600 gallery filenames all exist where the script will look.
import os, numpy as np
meta = np.load('embeddings/metadata.npz', allow_pickle=True)
caption_image_names = meta['caption_image_names'].tolist()
gallery_names = list(dict.fromkeys(caption_image_names))   # 1600 unique
IMG_DIR = 'data/Flickr8k_Dataset/Flicker8k_Dataset'
present = [n for n in gallery_names if os.path.isfile(os.path.join(IMG_DIR, n))]
print(f'Gallery filenames on disk: {len(present)}/{len(gallery_names)}')
if len(present) != len(gallery_names):
    missing = [n for n in gallery_names if n not in set(present)]
    print('MISSING (first 5):', missing[:5])
    # Show what IS on disk for debugging
    print('Files actually on disk (first 5):', sorted(os.listdir(IMG_DIR))[:5])
    raise SystemExit('Gallery files not found — fix image dir before rebuild.')
print('All 1,600 gallery images present on disk.')

## 5. Run the rebuild

This calls `rebuild_comparison_embeddings.py` which encodes the 1,600 captioned images (same order as `metadata.npz:image_names`) for both BLIP (ViT-L-14) and ALIGN (ViT-H-14), saves them as float32 with `image_names` embedded, and runs a sanity check (re-encodes image[0] and confirms cosine ≈ 1.0).

In [ ]:
# Smoke test first: encode 20 images, confirm sanity check passes.
!python rebuild_comparison_embeddings.py --model blip --limit 20 --check

In [ ]:
# Full build — BLIP + ALIGN, all 1,600 images + captions.
# On T4: ~1 min BLIP, ~3 min ALIGN.
!python rebuild_comparison_embeddings.py --model both --check

## 6. Verify before downloading

Confirm row count = 1600, cosine sanity ≥ 0.99, and the embeddings look reasonable (top-1 similarity for a known query should be a real match).

In [ ]:
import numpy as np
from model_comparison import load_blip_model, encode_single_text_model

blip = np.load('embeddings/blip_image_embeddings.npz', allow_pickle=True)['embeddings'].astype(np.float32)
names = np.load('embeddings/blip_image_embeddings.npz', allow_pickle=True)['image_names']
print('BLIP gallery shape:', blip.shape, '| row0 name:', names[0])
assert blip.shape[0] == 1600, f'Expected 1600 rows, got {blip.shape[0]}'

m, p, tok = load_blip_model()
for q in ['a dog on the beach', 'children playing football']:
    e = encode_single_text_model(q, m, tok)
    sims = (e @ blip.T).flatten()
    top = np.argsort(sims)[::-1][:3]
    print(f'\nBLIP {q!r}:')
    for i in top:
        print(f'  {names[i]}  sim={sims[i]:.4f}')

## 7. Download the rebuilt npz files

Copy these 4 files back into your local `embeddings/` folder (overwrite the old corrupt ones):
- `blip_image_embeddings.npz`
- `blip_text_embeddings.npz`
- `align_image_embeddings.npz`
- `align_text_embeddings.npz`

In [ ]:
from google.colab import files
for f in ['blip_image_embeddings.npz', 'blip_text_embeddings.npz',
          'align_image_embeddings.npz', 'align_text_embeddings.npz']:
    path = f'embeddings/{f}'
    if os.path.exists(path):
        print(f'Downloading {path} ({os.path.getsize(path)/1e6:.1f} MB) ...')
        files.download(path)

## 9. FP16 benchmark on CUDA (the missing Master-WPR number)

The local `benchmark.py` skips PyTorch FP16 on CPU because CPU fp16 has no tensor-core path — it would report a *slowdown*, which is misleading. The Master WPR's "FP16 ≈ 95 ms" claim is a **CUDA** number. This cell measures it properly on the T4, alongside a CUDA FP32 baseline, and saves a small JSON you merge into `latency.json` back home.

**You only need this if you want the FP16 row filled with a real number.** If you're fine framing FP16 as "CUDA-only, not benchmarked on CPU" in the report, skip this section entirely.

In [ ]:
# FP16 vs FP32 on CUDA T4 — 50 queries, same prompts as benchmark.py
import os, time, json
import numpy as np
import torch
import open_clip
from model_comparison import _l2_normalise  # reuse helper if available

N_QUERIES = 50
N_WARMUP = 5

SEED_QUERIES = [
    "a dog on the beach", "children playing football", "a woman cooking in the kitchen",
    "snow covered mountains", "a man riding a bike", "people sitting at a table",
    "a cat sitting on a chair", "a baby playing outdoors", "a red car parked on the street",
    "two birds flying in the sky", "a group of people hiking", "sunset over the ocean",
    "a black and white dog running", "kids playing in a park", "a person skiing downhill",
    "boats on a lake", "a girl holding an umbrella", "horses in a field",
    "a man climbing a rock wall", "city street at night",
]
queries = [SEED_QUERIES[i % len(SEED_QUERIES)] for i in range(N_QUERIES)]

assert torch.cuda.is_available(), 'T4 GPU not enabled — Runtime → Change runtime type → T4'
device = 'cuda'
print('Device:', torch.cuda.get_device_name(0))

# Load CLIP ViT-B/32 (openai) — matches model.py / export_onnx.py
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai', force_quick_gelu=True)
tokenizer = open_clip.get_tokenizer('ViT-B-32')

# Optional: inject fine-tuned light heads IF you uploaded light_clip_heads.pt.
# (Not required for the speedup measurement — keep it honest either way.)
if os.path.isfile('light_clip_heads.pt'):
    model.load_state_dict(torch.load('light_clip_heads.pt', map_location='cpu'), strict=False)
    print('Injected light_clip_heads.pt')
else:
    print('light_clip_heads.pt not uploaded — using base OpenAI weights (fine for speedup).')

def _stats(ms):
    a = np.asarray(ms, dtype=np.float64)
    return {'mean_ms': round(float(a.mean()),2), 'median_ms': round(float(np.median(a)),2),
            'p95_ms': round(float(np.percentile(a,95)),2), 'min_ms': round(float(a.min()),2),
            'max_ms': round(float(a.max()),2), 'std_ms': round(float(a.std()),2), 'n': int(a.size)}

def run_precision(dtype_str):
    model_d = model.to(device).eval()
    if dtype_str == 'fp16':
        model_d = model_d.half()
    dt = torch.float16 if dtype_str == 'fp16' else torch.float32

    # warmup
    with torch.no_grad():
        for q in queries[:N_WARMUP]:
            _ = model_d.encode_text(tokenizer([q]).to(device).to(dt))
    torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for q in queries:
            toks = tokenizer([q]).to(device).to(dt)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model_d.encode_text(toks)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000.0)
    return _stats(times)

fp32 = run_precision('fp32')
fp16 = run_precision('fp16')
speedup = round(fp32['mean_ms'] / fp16['mean_ms'], 2)

print('
' + '='*56)
print(f"{'Backend (CUDA T4)':<22} {'mean ms':>10} {'median':>10} {'p95':>10}")
print('-'*56)
print(f"{'PyTorch FP32':<22} {fp32['mean_ms']:>10.1f} {fp32['median_ms']:>10.1f} {fp32['p95_ms']:>10.1f}")
print(f"{'PyTorch FP16':<22} {fp16['mean_ms']:>10.1f} {fp16['median_ms']:>10.1f} {fp16['p95_ms']:>10.1f}")
print('-'*56)
print(f"FP16 speedup vs FP32: {speedup}x")

out = {
    'device': torch.cuda.get_device_name(0),
    'n_queries': N_QUERIES,
    'metric': 'CLIP ViT-B/32 text encode latency (ms), single query, CUDA',
    'fp32_cuda': fp32,
    'fp16_cuda': fp16,
    'fp16_speedup_vs_fp32': speedup,
    'note': 'Measured on Colab T4. Merge into embeddings/latency.json via: python benchmark.py --merge-fp16 fp16_cuda_benchmark.json',
}
with open('fp16_cuda_benchmark.json', 'w') as f:
    json.dump(out, f, indent=2)
print('
Saved -> fp16_cuda_benchmark.json')

## 8. Back on your laptop

After downloading the 4 npz files into `C:\Users\rde48\Desktop\image-search-app\embeddings\`, run:

```bash
venv\Scripts\python.exe evaluate.py
```

You should now see BLIP P@5 climb from 0% to 40–80%, matching the Master WPR table. The `precision_scores.json` will be regenerated.